# Pose Transformer inference

Curated portfolio copy of the original coursework notebook. Run cells in order with the required data and dependencies.


## Environment


In [ ]:
# ===== INSTALL DEPENDENCIES =====
!pip install huggingface_hub
!pip install boto3 -q
!pip install opencv-python torch numpy torchvision tqdm
!pip install mediapipe


In [ ]:
# Import the required libraries
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import os
import cv2
import numpy as np
from tqdm import tqdm
import time
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import requests
import math



## Model account


In [ ]:
hf_username = "zz-xu"

## Model architecture


In [ ]:
# =============================================================================
# 1. MODEL DEFINITION (must match training)
# =============================================================================

import torch
import torch.nn as nn
from huggingface_hub import HfApi, hf_hub_download

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=64, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Register as buffer
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # x: [Batch, Sequence_Len, Feature_Dim]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

class PushUpTransformer(nn.Module):
    def __init__(self, input_dim=72, num_classes=10, d_model=128, nhead=4, num_layers=3, dropout=0.2):
        super().__init__()

        # 1. Feature Projection & Normalization
        self.embedding = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

        # 2. Positional Encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=64, dropout=dropout)

        # 3. Transformer Encoder Body
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=512,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # Input x shape: (Batch_Size, 64, 72)

        # Embed and Add Position
        x = self.embedding(x)      # -> (Batch, 64, 128)
        x = self.pos_encoder(x)    # -> (Batch, 64, 128)

        # Transformer Pass (Self-Attention)
        x = self.transformer(x)    # -> (Batch, 64, 128)

        # Global Average Pooling
        x = x.mean(dim=1)          # -> (Batch, 128)

        # Classify
        logits = self.classifier(x) # -> (Batch, num_classes)
        return logits


## Dataset and pose features


In [ ]:
# =============================================================================
# DOWNLOAD TEST DATA FROM S3
# =============================================================================

def download_test_data(bucket_name='training-and-validation-data',download_dir='./test-data'):
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

    bucket_name = 'prism-mvta'
    prefix = 'training-and-validation-data/'

    os.makedirs(download_dir, exist_ok=True)

    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    video_names = []

    for page in pages:
        if 'Contents' not in page:
            print("No files found at the specified path!")
            break

        print("Downloading test data:\n")
        for obj in tqdm(page['Contents']):
            key = obj['Key']
            filename = os.path.basename(key)

            if not filename:
                continue

            video_names.append(filename)
            local_path = os.path.join(download_dir, filename)
            # print(f"Downloading: {filename}")
            s3.download_file(bucket_name, key, local_path)

    print(f"\nDownloaded {len(video_names)} test videos")
    return download_dir



In [ ]:
# ============================================================================= # DATASET AND DATALOADER =============================================================================


def extract_and_save_poses(video_dir, output_dir):

    # Download mediapipe model
    model_path = 'pose_landmarker_full.task'
    url = "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task"
    if not os.path.exists(model_path):
        print(f"Downloading MediaPipe Model")
        with open(model_path, 'wb') as f: f.write(requests.get(url).content)

    # MediaPipe Setup
    BaseOptions = mp.tasks.BaseOptions
    PoseLandmarker = mp.tasks.vision.PoseLandmarker
    PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    os.makedirs(output_dir, exist_ok=True)

    # Configure Landmarker for VIDEO mode
    options = PoseLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.VIDEO
    )

    with PoseLandmarker.create_from_options(options) as landmarker:
        video_files = list(Path(video_dir).glob('*.mp4'))

        for video_path in video_files:
            output_path = Path(output_dir) / f"{video_path.stem}.npz"
            if output_path.exists():
                # print(f"Skip (already exists): {output_path.name}")
                continue

            print(f"Processing: {video_path.name}")
            cap = cv2.VideoCapture(str(video_path))
            fps = cap.get(cv2.CAP_PROP_FPS)

            all_frames_data = []
            frame_count = 0

            with vision.PoseLandmarker.create_from_options(options) as landmarker:
                while cap.isOpened():
                    ret, frame = cap.read()
                    if not ret: break

                    # RGB
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

                    # Calculate timestamp
                    timestamp_ms = int((frame_count * 1000) / fps)

                    # Detect
                    result = landmarker.detect_for_video(mp_image, timestamp_ms)

                    # 33 joints * (x, y, z, visibility)
                    frame_coords = np.zeros((33, 4))
                    if result.pose_landmarks:
                        for i, lm in enumerate(result.pose_landmarks[0]):
                            frame_coords[i] = [lm.x, lm.y, lm.z, lm.presence]

                    all_frames_data.append(frame_coords)
                    frame_count += 1

            cap.release()

            # Save as compressed npz
            np.savez_compressed(
                output_path,
                coords=np.array(all_frames_data, dtype=np.float32), #(T,33,4)
                fps=np.array(fps)
            )
            # print(f"Saved {frame_count} frames")
    print('Extraction complete')

def build_valset(
    input_dir: str,
    val_files: list[str],
    output_dir: str,
    L: int = 64,
):
    """
    Build validation set by UNIFORM sampling L frames per video.
    """
    input_dir = str(input_dir)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    saved = 0

    for f in val_files:
        label = int(f.split("_")[0])
        data = np.load(os.path.join(input_dir, f))
        coords = data["coords"].astype(np.float32)  # (T,33,4)
        T = coords.shape[0]

        # Uniform deterministic sampling with padding
        if T <= 0:
            clip = np.zeros((L, 33, 4), dtype=np.float32)
            idx = np.zeros((L,), dtype=np.int64)
        elif T >= L:
            idx = np.linspace(0, T - 1, L, dtype=np.int64)
            clip = coords[idx]
        else:
            # T < L: take all frames then pad by repeating last
            pad = np.repeat(coords[-1][None, :, :], repeats=(L - T), axis=0)
            clip = np.concatenate([coords, pad], axis=0)
            idx = np.concatenate([np.arange(T, dtype=np.int64), np.full(L - T, T - 1, dtype=np.int64)])

        out_path = output_dir / f"{label}__val__{Path(f).stem}.npz"
        np.savez_compressed(
            out_path,
            clip=clip.astype(np.float32),   # (L,33,4)
            label=int(label),
            method="uniform",
            source=Path(f).stem,
            idx=idx,
        )
        saved += 1

    print(f"Saved {saved} clips to: {output_dir}")

import torch
from torch.utils.data import Dataset

class SkeletonDataset(Dataset):
    def __init__(self, data_dir, is_train=True, L=64):
        self.data_dir = data_dir
        self.file_list = [f for f in os.listdir(data_dir) if f.endswith('.npz')]
        self.is_train = is_train
        self.L = L

        # Indices in MediaPipe:
        # Shoulders (11,12), Elbows (13,14), Wrists (15,16), Hips (23,24)
        self.selected_joints = [11, 12, 13, 14, 15, 16, 23, 24]

    def __len__(self):
        return len(self.file_list)

    def _apply_augmentation(self, traj):

        # 1. Random Horizontal Flip (Mirroring)
        if np.random.rand() > 0.5:
            traj[:, :, 0] = 1.0 - traj[:, :, 0]
            left = [11, 13, 15, 23]
            right = [12, 14, 16, 24]
            traj[:, left + right] = traj[:, right + left]

        # 2. Random Rotation
        angle = np.radians(np.random.uniform(-10, 10))
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        x, y = traj[:, :, 0].copy(), traj[:, :, 1].copy()
        traj[:, :, 0] = x * cos_a - y * sin_a
        traj[:, :, 1] = x * sin_a + y * cos_a

        # 3. Random Scaling (Zoom in/out)
        scale = np.random.uniform(0.9, 1.1)
        traj[:, :, :3] *= scale

        return traj

    def _calculate_angle(self, a, b, c):
        """Calculates the angle at joint 'b' given points a, b, and c."""
        ba = a - b
        bc = c - b
        norm_ba = np.linalg.norm(ba, axis=-1, keepdims=True)
        norm_bc = np.linalg.norm(bc, axis=-1, keepdims=True)

        cosine_angle = np.sum(ba * bc, axis=-1, keepdims=True) / (norm_ba * norm_bc + 1e-6)
        angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
        return angle

    def _robust_center(self, feats):
        """
        Centering Logic: Hips -> Shoulders -> Global Mean
        """
        hip_vis = (feats[:, 6, 3] + feats[:, 7, 3]) / 2.0
        shld_vis = (feats[:, 0, 3] + feats[:, 1, 3]) / 2.0

        if np.mean(hip_vis) > 0.5:
            center = (feats[:, 6, :3] + feats[:, 7, :3]) / 2.0
        elif np.mean(shld_vis) > 0.5:
            center = (feats[:, 0, :3] + feats[:, 1, :3]) / 2.0
        else:
            center = np.mean(feats[:, :, :3], axis=1)

        feats[:, :, :3] -= center[:, np.newaxis, :]
        return feats

    def _engineer_features(self, traj):
        """
        1. Centering
        2. Angle Calculation (Elbows & Shoulders)
        3. Velocity calculation
        """
        # Slice joints
        # Map: 0:L_sh, 1:R_sh, 2:L_el, 3:R_el, 4:L_wr, 5:R_wr, 6:L_hip, 7:R_hip
        feats = traj[:, self.selected_joints, :].copy()

        # 1. Apply Robust Centering
        feats = self._robust_center(feats)

        # 2. Compute Angles
        # Left Elbow (Shoulder-Elbow-Wrist)
        l_elbow_angle = self._calculate_angle(feats[:,0,:3], feats[:,2,:3], feats[:,4,:3])
        # Right Elbow
        r_elbow_angle = self._calculate_angle(feats[:,1,:3], feats[:,3,:3], feats[:,5,:3])
        # Left Shoulder (Hip-Shoulder-Elbow)
        l_shld_angle = self._calculate_angle(feats[:,6,:3], feats[:,0,:3], feats[:,2,:3])
        # Right Shoulder
        r_shld_angle = self._calculate_angle(feats[:,7,:3], feats[:,1,:3], feats[:,3,:3])

        # 3. Assemble Feature Vector per frame
        # Positions (24) + Visibilities (8) + Angles (4) = 36 features
        pos_vis = feats.reshape(self.L, -1) # Flatten x,y,z,v
        angles = np.concatenate([l_elbow_angle, r_elbow_angle, l_shld_angle, r_shld_angle], axis=-1)

        base_features = np.concatenate([pos_vis, angles], axis=-1) # (L, 36)

        # 4. Velocity Features
        velocity = np.zeros_like(base_features)
        velocity[1:] = base_features[1:] - base_features[:-1]

        # Final Vector
        return np.concatenate([base_features, velocity], axis=-1)

    def __getitem__(self, idx):
        data = np.load(os.path.join(self.data_dir, self.file_list[idx]))
        traj = data['clip'].astype(np.float32)
        label = int(self.file_list[idx].split('_')[0])

        if self.is_train:
            traj = self._apply_augmentation(traj)

        features = self._engineer_features(traj) # (64, 72)
        return torch.tensor(features), torch.tensor(label, dtype=torch.long)



## Checkpoint download


In [ ]:
# =============================================================================
# DOWNLOAD MODEL FROM HUGGING FACE
# =============================================================================

def load_model_from_hub(repo_id, num_classes=12):
    model_path = hf_hub_download(repo_id=repo_id, filename="model.pt")

    model = PushUpTransformer(num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location='cpu'))

    print(f"Model loaded from {repo_id}")
    return model

model = load_model_from_hub(f"{hf_username}/mv-final-assignment",num_classes=12)



## Evaluation


In [ ]:
def evaluate(model, test_loader, dataset, device):
    model.eval()
    correct = 0
    total = 0

    all_preds = []
    all_labels = []
    all_times = []

    print("\n")

    with torch.no_grad():
        for idx, (frames, labels) in enumerate(test_loader):
            frames, labels = frames.to(device), labels.to(device)

            # Time the forward pass
            start_time = time.time()
            outputs = model(frames)
            if device.type == 'cuda':
                torch.cuda.synchronize()  # wait for GPU to finish
            end_time = time.time()

            inference_time = (end_time - start_time) * 1000  # ms
            all_times.append(inference_time)

            preds = outputs.argmax(dim=1) + 1

            for i in range(labels.size(0)):
                batch_idx = idx * test_loader.batch_size + i
                video_name = dataset.file_list[batch_idx]
                pred = preds[i].item()
                true_label = labels[i].item()
                is_correct = "✓" if pred == true_label else "✗"

                print(f"{is_correct}  pred={pred}  true={true_label}  |  {inference_time:>7.1f}ms  |  {video_name}")

            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = correct / total
    return accuracy, all_preds, all_labels, all_times


# =============================================================================
# RUN INFERENCE
# =============================================================================

def run_inference(model, bucket_name='training-and-validation-data'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Download test data
    test_dir = download_test_data(bucket_name, './test-data')

    model = model.to(device)

    # Create dataloader
    extract_and_save_poses(test_dir, './mediapipe-skeleton')
    val_files = [p.name for p in Path('./mediapipe-skeleton').iterdir() if p.is_file()]
    build_valset('mediapipe-skeleton', val_files=val_files, output_dir='test_data_processed')


    test_dataset = SkeletonDataset('test_data_processed')
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
    )

    print(f"\nRunning inference on {len(test_dataset)} test videos...")

    # Warmup (optional, helps get consistent GPU timings)
    if device.type == 'cuda':
        dummy = torch.randn(1, 64, 72).to(device)
        with torch.no_grad():
            _ = model(dummy)
        torch.cuda.synchronize()

    total_start = time.time()
    accuracy, preds, labels, times = evaluate(model, test_loader, test_dataset, device)
    total_end = time.time()

    # Summary
    num_correct = sum(p == l for p, l in zip(preds, labels))
    num_wrong = len(preds) - num_correct

    print("\n" + "="*50)
    print("SUMMARY")
    print("="*50)
    print(f"Total videos:         {len(preds)}")
    print(f"Correct:              {num_correct}")
    print(f"Incorrect:                {num_wrong}")
    print(f"")
    print(f"ACCURACY:             {accuracy*100:.2f}%")
    print(f"")
    print(f"Total time:           {total_end - total_start:.2f}s")
    print(f"Avg per video:        {sum(times) / len(times):.1f}ms")
    print(f"Min latency:          {min(times):.1f}ms")
    print(f"Max latency:          {max(times):.1f}ms")
    print("="*50)
    return accuracy, preds, labels

_, _, _ = run_inference(model)